In [1]:
import pandas as pd 
import json
import os
import ast

In [2]:
query_data = pd.read_json('data\MultiHopRAG.json')
query_data

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\andre\AppData\Local\Temp\ipykernel_9040\1842584137.py:1: SyntaxWarning: invalid escape sequence '\M'
  query_data = pd.read_json('data\MultiHopRAG.json')


,query,answer,question_type,evidence_list
0,Who is the individual associated with the cryp...,Sam Bankman-Fried,inference_query,[{'title': 'The FTX trial is bigger than Sam B...
1,Which individual is implicated in both inflati...,Donald Trump,inference_query,[{'title': 'Donald Trump defrauded banks with ...
2,Who is the figure associated with generative A...,Sam Altman,inference_query,[{'title': 'OpenAI's ex-chairman accuses board...
3,Do the TechCrunch article on software companie...,Yes,comparison_query,"[{'title': 'Here’s how Rainforest, a budding S..."
4,Which online betting platform provides a welco...,Caesars Sportsbook,inference_query,[{'title': '2023 Kentucky online sports bettin...
...,...,...,...,...
2551,Who is the individual associated with using FT...,Sam Bankman-Fried,inference_query,[{'title': 'Sam Bankman-Fried was a terrible b...
2552,Who is the individual that was once likened to...,Sam Bankman-Fried,inference_query,"[{'title': 'SBF’s trial starts soon, but how d..."
2553,"Who, according to articles in Sporting News, s...",Bettors,inference_query,[{'title': 'Moneyline Betting - What is the Mo...
2554,Does the Cnbc | World Business News Leader art...,Yes,comparison_query,"[{'title': 'Sex, Signal messages, and sabotagi..."


In [ ]:
fixed_results = pd.DataFrame()
recursive_results = pd.DataFrame()
semantic_results = pd.DataFrame()

for i in range(len(os.listdir('results')) // 3 - 1):
    temp_fixed = pd.read_csv(f'results\\fixed_results_qa_part_{i+1}.csv')
    temp_recursive = pd.read_csv(f'results\\recursive_results_qa_part_{i+1}.csv')
    temp_semantic = pd.read_csv(f'results\\semantic_results_qa_part_{i+1}.csv')

    fixed_results = pd.concat([fixed_results, temp_fixed], ignore_index=True)   
    recursive_results = pd.concat([recursive_results, temp_recursive], ignore_index=True)
    semantic_results = pd.concat([semantic_results, temp_semantic], ignore_index=True)
    
fixed_results.to_csv('results/fixed_results_qa.csv', index=False)
semantic_results.to_csv('results/semantic_results_qa.csv', index=False)
recursive_results.to_csv('results/recursive_results_qa.csv', index=False)

100 100 100
100 100 100


In [4]:
def has_intersection(a, b):
    a_words = set(a.split())
    b_words = set(b.split())
    return len(a_words.intersection(b_words)) > 0


def calculate_qa_metrics(pred_list, gold_list):
    tp = sum(1 for pred, gold in zip(pred_list, gold_list) if has_intersection(pred.lower(), gold.lower()))
    fp = sum(1 for pred, gold in zip(pred_list, gold_list) if not has_intersection(pred.lower(), gold.lower()))
    fn = len(gold_list) - tp
    tn = len(pred_list) - tp 

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0

    return precision, recall, f1, accuracy

In [5]:
gold_answers = query_data['answer'][:200].tolist()
fixed_predictions = fixed_results['answer'].tolist()
semantic_predictions = semantic_results['answer'].tolist()
recursive_predictions = recursive_results['answer'].tolist()

qa_results = []
for name, lst in {'fixed': fixed_predictions, 'semantic': semantic_predictions, 'recursive': recursive_predictions}.items():
    precision, recall, f1, accuracy = calculate_qa_metrics(lst, gold_answers)
    qa_results.append({
        'chunking_strategy': name,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'accuracy': accuracy
    })
qa_results = pd.DataFrame(qa_results)
qa_results.to_csv('results/qa_evaluation_metrics.csv', index=False)

In [6]:
def calculate_retrieval_metrics(retrieved_lists, gold_lists, k=4):
    hits_at_k_count = 0
    map_at_k_list = []
    mrr_list = []

    for retrieved, gold in zip(retrieved_lists, gold_lists):
        hits_at_k_flag = False
        average_precision_sum = 0
        first_relevant_rank = None
        find_gold = []

        gold = [item.replace(" ", "").replace("\n", "") for item in gold]
        retrieved = [item.replace(" ", "").replace("\n", "") for item in retrieved]
    
        for rank, retrieved_item in enumerate(retrieved[:k], start=1):
            if any(gold_item in retrieved_item for gold_item in gold):
                if not hits_at_k_flag:
                    hits_at_k_flag = True
                if first_relevant_rank is None:
                    first_relevant_rank = rank

                count = 0
                for gold_item in gold:
                    if gold_item in retrieved_item and gold_item not in find_gold:
                        count += 1
                        find_gold.append(gold_item)
                precision_at_rank = count / rank
                average_precision_sum += precision_at_rank

        hits_at_k_count += int(hits_at_k_flag)
        map_at_k_list.append(average_precision_sum / min(len(gold), k))
        mrr_list.append(1 / first_relevant_rank if first_relevant_rank else 0)

    hits_at_k = hits_at_k_count / len(gold_lists)
    map_at_k = sum(map_at_k_list) / len(gold_lists)
    mrr_at_k = sum(mrr_list) / len(gold_lists)

    return {
        f'Hits@{k}': hits_at_k,
        f'MAP@{k}': map_at_k,
        f'MRR@{k}': mrr_at_k,
    }

In [7]:
gold_contexts = query_data['evidence_list'].apply(lambda x: [i['fact'] for i in x])[:200].tolist()

fixed_contexts = fixed_results['retrieved_chunks'].apply(ast.literal_eval).tolist()
semantic_contexts = semantic_results['retrieved_chunks'].apply(ast.literal_eval).tolist()
recursive_contexts = recursive_results['retrieved_chunks'].apply(ast.literal_eval).tolist()

df = query_data[:200]
null_index = df[df['question_type'] == 'null_query'].index

gold_contexts = [gold_contexts[i] for i in range(200) if i not in null_index]
fixed_contexts = [fixed_contexts[i] for i in range(200) if i not in null_index]
semantic_contexts = [semantic_contexts[i] for i in range(200) if i not in null_index]
recursive_contexts = [recursive_contexts[i] for i in range(200) if i not in null_index]

retrieval_results = []
for name, lst in {'fixed': fixed_contexts, 'semantic': semantic_contexts, 'recursive': recursive_contexts}.items():
    metrics = calculate_retrieval_metrics(lst, gold_contexts)
    retrieval_results.append({
        'chunking_strategy': name,
        **metrics
    })

retrieval_results = pd.DataFrame(retrieval_results)
retrieval_results.to_csv('results/retrieval_evaluation_metrics.csv', index=False)